# Part 2: Web Search — Inverted Index
**Goal:** Build an inverted index over a collection of webpages and answer search queries  


In [ ]:
import os
import math

# These small connector words carry no meaning about content,
# so we skip storing them in the index (but still COUNT them for position tracking)
STOP_WORDS = {
    'a', 'an', 'the', 'they', 'these', 'this',
    'for', 'is', 'are', 'was', 'of', 'or',
    'and', 'does', 'will', 'whose'
}

# Characters that should be treated as word separators / whitespace
PUNCTUATION = set('{}[]<>=(). ,;\'"?#!-:')

# Plural -> singular mappings (treat both forms as the same word)
# The assignment says to consider only the provided list as exhaustive
PLURAL_MAP = {
    'stacks'       : 'stack',
    'structures'   : 'structure',
    'applications' : 'application',
    'pages'        : 'page',
    'words'        : 'word',
    'indices'      : 'index',
    'indexes'      : 'index',
    'entries'      : 'entry',
    'queries'      : 'query',
    'engineers'    : 'engineer',
    'trees'        : 'tree',
    'nodes'        : 'node',
    'edges'        : 'edge',
    'graphs'       : 'graph',
    'lists'        : 'list',
    'arrays'       : 'array',
    'queues'       : 'queue',
    'tables'       : 'table',
    'keys'         : 'key',
    'values'       : 'value',
    'pointers'     : 'pointer',
    'algorithms'   : 'algorithm',
    'computers'    : 'computer',
    'programs'     : 'program',
    'functions'    : 'function',
    'methods'      : 'method',
    'classes'      : 'class',
    'objects'      : 'object',
    'files'        : 'file',
    'searches'     : 'search',
    'sorts'        : 'sort',
}

WEBPAGES_FOLDER = "Q2_WebSearch/webpages"
ACTIONS_FILE    = "Q2_WebSearch/actions.txt"
ANSWERS_FILE    = "Q2_WebSearch/answers.txt"

print("Constants defined.")

Constants defined.


### normalize() — Text Cleaning

- Core preprocessing function for all words  
- Applies: lowercase conversion, punctuation removal, plural → singular mapping  
- Used before storing or searching words  
- Returns empty string `''` if the token becomes empty  

In [ ]:
def normalize(word):
    """
    Clean a raw token:
      1. Lowercase it
      2. Strip punctuation characters
      3. Map plural to singular if in PLURAL_MAP
    Returns the cleaned string (may be empty if token was pure punctuation).
    """
    word = word.lower()
    # Replace each punctuation character with empty string
    for ch in PUNCTUATION:
        word = word.replace(ch, '')
    word = word.strip()
    # Apply plural -> singular mapping
    return PLURAL_MAP.get(word, word)

# Quick sanity checks
print(normalize("Structures."))   # should print: structure
print(normalize("DATA"))          # should print: data
print(normalize("!!!"))           # should print: (empty)
print(normalize("engineers,"))    # should print: engineer

structure
data

engineer


### Position Class

- Represents the location of a word  
- Stores:
  - Page where the word appears  
  - Word index (1-based position in the token list)  

In [ ]:
class Position:
    def __init__(self, page_entry, word_index):
        # page_entry: the PageEntry object this position belongs to
        # word_index: 1-based index in the original token stream
        self._page  = page_entry
        self._index = word_index

    def getPageEntry(self):
        return self._page

    def getWordIndex(self):
        return self._index

    def __repr__(self):
        return f"Position(page={self._page.name}, idx={self._index})"

print("Position class defined.")

Position class defined.


### WordEntry Class

- Stores all positions of a specific word  
- Tracks occurrences across one or more documents  
- Computes term frequency (TF) for use in TF-IDF scoring 

In [ ]:
class WordEntry:
    def __init__(self, word):
        self.word      = word
        self.positions = []  # list of Position objects

    def addPosition(self, position):
        """Add a single Position entry."""
        self.positions.append(position)

    def addPositions(self, position_list):
        """Add multiple Position entries at once."""
        self.positions.extend(position_list)

    def getAllPositionsForThisWord(self):
        """Return the full list of positions for this word."""
        return self.positions

    def getTermFrequency(self, page_entry):
        """
        TF = (occurrences of this word in this page) / (total words in this page)
        Only counts positions belonging to the given page.
        """
        count = sum(1 for p in self.positions if p.getPageEntry() is page_entry)
        total_words = page_entry.getTotalWordCount()
        if total_words == 0:
            return 0.0
        return count / total_words

    def __repr__(self):
        return f"WordEntry('{self.word}', {len(self.positions)} positions)"

print("WordEntry class defined.")

WordEntry class defined.


### PageIndex Class

- Represents the index for a single document  
- Maps each unique (non-stop) word to a WordEntry  
- Similar to an index at the back of a book

In [ ]:
class PageIndex:
    def __init__(self):
        # word string -> WordEntry object
        self._entries = {}

    def addPositionForWord(self, word, position):
        """
        Add a position for the given word.
        Creates a new WordEntry if this word hasn't been seen before.
        """
        if word not in self._entries:
            self._entries[word] = WordEntry(word)
        self._entries[word].addPosition(position)

    def getWordEntries(self):
        """Return all WordEntry objects in this page's index."""
        return list(self._entries.values())

    def getWordEntry(self, word):
        """Return the WordEntry for a specific word, or None if not found."""
        return self._entries.get(word, None)

print("PageIndex class defined.")

PageIndex class defined.


### PageEntry Class

- Represents a single webpage  
- On initialization:
  - Reads the file  
  - Tokenizes and normalizes words  
  - Builds the page’s local index (PageIndex)  

- Position indexing:
  - All tokens (including stop words) are counted for positions  
  - Only non-stop words are included in the index  

In [ ]:
class PageEntry:
    def __init__(self, page_name, folder=None):
        self.name = page_name
        self._index      = PageIndex()
        self._total_words = 0  # total raw token count (for TF denominator)

        folder = folder or WEBPAGES_FOLDER
        filepath = os.path.join(folder, page_name)

        with open(filepath, 'r', errors='ignore') as f:
            content = f.read()

        # Split on whitespace to get raw tokens
        raw_tokens = content.split()
        self._total_words = len(raw_tokens)  # all tokens, including stop words

        for i, token in enumerate(raw_tokens):
            word = normalize(token)

            # Skip empty strings (pure punctuation tokens)
            if not word:
                continue

            # Skip stop words — don't index them
            # but their position (i+1) is already counted in total_words above
            if word in STOP_WORDS:
                continue

            # i+1 because positions are 1-based
            pos = Position(self, i + 1)
            self._index.addPositionForWord(word, pos)

    def getPageIndex(self):
        return self._index

    def getTotalWordCount(self):
        """Total number of tokens in this page (used for TF calculation)."""
        return self._total_words

    def __repr__(self):
        return f"PageEntry('{self.name}')"

print("PageEntry class defined.")

PageEntry class defined.


### InvertedPageIndex Class

- Global index across all pages  
- Maps each word to all (page, position) occurrences  
- Combines data from individual PageIndex objects  
- Updated by merging a page’s local index when added  

In [ ]:
class InvertedPageIndex:
    def __init__(self):
        # List of all PageEntry objects added so far
        self._pages = []
        # Global word -> list of Position objects (across ALL pages)
        self._inv   = {}

    def addPage(self, page_entry):
        """
        Add a new page to the inverted index.
        Merges the page's local index into the global inverted index.
        """
        self._pages.append(page_entry)

        # Walk through every word in this page's index
        for word_entry in page_entry.getPageIndex().getWordEntries():
            word = word_entry.word
            if word not in self._inv:
                self._inv[word] = []
            # Add all positions for this word from this page
            self._inv[word].extend(word_entry.getAllPositionsForThisWord())

    def getPagesWhichContainWord(self, word):
        """
        Return a set of PageEntry objects that contain the given word.
        Returns empty set if word not found.
        """
        if word not in self._inv:
            return set()
        return {pos.getPageEntry() for pos in self._inv[word]}

    def getPositionsOfWordInPage(self, word, page_entry):
        """
        Return sorted list of word indices where 'word' appears in 'page_entry'.
        """
        if word not in self._inv:
            return []
        return sorted(
            pos.getWordIndex()
            for pos in self._inv[word]
            if pos.getPageEntry() is page_entry
        )

    def getAllPages(self):
        return self._pages

    def getPageByName(self, name):
        """Look up a page by filename. Returns None if not found."""
        for p in self._pages:
            if p.name == name:
                return p
        return None

print("InvertedPageIndex class defined.")

InvertedPageIndex class defined.


### SearchEngine Class

- Main controller that integrates all components  
- Parses action strings and calls appropriate methods  

- Supported actions:
  - `addPage <filename>`  
  - `queryFindPagesWhichContainWord <word>`  
  - `queryFindPositionsOfWordInAPage <word> <page>`  

In [ ]:
class SearchEngine:
    def __init__(self):
        self._ipi = InvertedPageIndex()

    def performAction(self, action_message):
        """
        Parse and execute one action string.
        Returns the output string (also prints it).
        """
        parts = action_message.strip().split()
        if not parts:
            return

        cmd = parts[0]

        # ── addPage ──────────────────────────────────────────────────
        if cmd == "addPage":
            page_name  = parts[1]
            page_entry = PageEntry(page_name)
            self._ipi.addPage(page_entry)
            # addPage produces no printed output per the assignment spec

        # ── queryFindPagesWhichContainWord ───────────────────────────
        elif cmd == "queryFindPagesWhichContainWord":
            raw_word = parts[1]
            word     = normalize(raw_word)
            pages    = self._ipi.getPagesWhichContainWord(word)

            if pages:
                # Sort alphabetically for consistent output
                result = ", ".join(sorted(p.name for p in pages))
                print(result)
            else:
                print(f"No webpage contains word {raw_word}")

        # ── queryFindPositionsOfWordInAPage ──────────────────────────
        elif cmd == "queryFindPositionsOfWordInAPage":
            raw_word  = parts[1]
            page_name = parts[2]
            word      = normalize(raw_word)

            # First check: does this page exist in our database?
            page_entry = self._ipi.getPageByName(page_name)
            if page_entry is None:
                print(f"No webpage {page_name} found")
                return

            # Page exists — now check if the word is in it
            positions = self._ipi.getPositionsOfWordInPage(word, page_entry)
            if positions:
                print(", ".join(str(x) for x in positions))
            else:
                print(f"Webpage {page_name} does not contain word {raw_word}")

        else:
            print(f"Unknown action: {cmd}")

print("SearchEngine class defined.")

SearchEngine class defined.


### Run Search Engine

- Read each line from `actions.txt`  
- Execute actions using the SearchEngine  
- Compare output with `answers.txt` for verification  

In [ ]:
print("Running actions from actions.txt...")
print("=" * 55)

engine = SearchEngine()

with open(ACTIONS_FILE, 'r') as f:
    for line in f:
        line = line.strip()
        if line:  # skip blank lines
            engine.performAction(line)

print("=" * 55)
print("Done. Compare output above with answers.txt")

Running actions from actions.txt...
No webpage contains word delhi
stack_datastructure_wiki
stack_datastructure_wiki
Webpage stack_datastructure_wiki does not contain word magazines
No webpage contains word allain
stack_cprogramming
stack_cprogramming
stack_cprogramming
stack_oracle
stack_cprogramming, stack_datastructure_wiki, stackoverflow
stackmagazine
Done. Compare output above with answers.txt


### Verification — Output Comparison

- Run the engine and capture output  
- Compare with `answers.txt`  
- Green lines indicate correct matches  
- Red lines indicate mismatches  

In [ ]:
import io
import sys

# Capture printed output
captured = io.StringIO()
sys.stdout = captured

engine2 = SearchEngine()
with open(ACTIONS_FILE, 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            engine2.performAction(line)

sys.stdout = sys.__stdout__  # restore normal stdout

our_output  = [l for l in captured.getvalue().splitlines() if l.strip()]
with open(ANSWERS_FILE, 'r') as f:
    expected = [l.strip() for l in f.readlines() if l.strip()]

print(f"Our output lines   : {len(our_output)}")
print(f"Expected answer lines: {len(expected)}")
print()

all_correct = True
for i, (got, want) in enumerate(zip(our_output, expected)):
    if got == want:
        print(f"  Line {i+1:3d} ✓  {got}")
    else:
        print(f"  Line {i+1:3d} ✗")
        print(f"    Expected : {want}")
        print(f"    Got      : {got}")
        all_correct = False

print()
if all_correct:
    print("All answers match!")
else:
    print("Some answers differ — check normalize() and PLURAL_MAP.")

Our output lines   : 11
Expected answer lines: 11

  Line   1 ✓  No webpage contains word delhi
  Line   2 ✓  stack_datastructure_wiki
  Line   3 ✓  stack_datastructure_wiki
  Line   4 ✓  Webpage stack_datastructure_wiki does not contain word magazines
  Line   5 ✓  No webpage contains word allain
  Line   6 ✓  stack_cprogramming
  Line   7 ✓  stack_cprogramming
  Line   8 ✓  stack_cprogramming
  Line   9 ✓  stack_oracle
  Line  10 ✓  stack_cprogramming, stack_datastructure_wiki, stackoverflow
  Line  11 ✓  stackmagazine

All answers match!
